# Healthcare Data Analysis for Disease Risk Prediction
**Educational demo:** synthetic patient data and rule-generated labels; not real diagnoses.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
df = pd.read_csv('../dataset/healthcare_risk_synthetic.csv')
df.head()

## 1. Data inspection and cleaning

In [ ]:
print('Shape:', df.shape)
print(df.info())
print('Missing values:\n', df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().copy()
df.describe(include='all')

## 2. Visualizations

In [ ]:
df['Risk_Label'].value_counts().plot(kind='bar', title='Synthetic Risk Label Counts')
plt.xlabel('Label'); plt.ylabel('Records'); plt.tight_layout(); plt.show()

In [ ]:
sns.histplot(data=df, x='Age', hue='Risk_Label', bins=15, multiple='stack')
plt.title('Age Distribution by Synthetic Label'); plt.tight_layout(); plt.show()

In [ ]:
numeric=['Age','BMI','Systolic_BP','Glucose','Cholesterol']
sns.heatmap(df[numeric].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap'); plt.tight_layout(); plt.show()

## 3. Baseline machine-learning model
This predicts the synthetic label only. Do not use for diagnosis or treatment decisions.

In [ ]:
X=df.drop(columns=['Patient_ID','Risk_Label']); y=df['Risk_Label']
cat_cols=X.select_dtypes(include='object').columns.tolist(); num_cols=X.select_dtypes(exclude='object').columns.tolist()
preprocess=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols)])
model=Pipeline([('prep',preprocess),('clf',RandomForestClassifier(n_estimators=150,random_state=42,class_weight='balanced'))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
model.fit(X_train,y_train); pred=model.predict(X_test)
print('Accuracy:',round(accuracy_score(y_test,pred),3))
print(classification_report(y_test,pred,zero_division=0))
print('Confusion matrix:\n',confusion_matrix(y_test,pred,labels=['Low','High']))

## Conclusion
This workflow demonstrates data loading, cleaning checks, visualization, and model evaluation. Synthetic labels cannot establish real-world clinical accuracy.